In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
if (cwd / "src").exists():
    project_root = cwd
elif (cwd.parent / "src").exists():
    project_root = cwd.parent
else:
    raise FileNotFoundError("Could not locate project root containing 'src'.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
import statsmodels.api as sm

from src.data.create_dataframes import processed_df_map

pd.set_option("display.float_format", "{:,.4f}".format)

DEFAULT_Y_COL_MAP = {
    "eur_df_processed": "EURUSD",
    "gbp_df_processed": "GBPUSD",
    "jpy_df_processed": "USDJPY",
    "chf_df_processed": "USDCHF",
    "cad_df_processed": "USDCAD",
    "aud_df_processed": "AUDUSD",
    "nzd_df_processed": "NZDUSD",
    "nok_df_processed": "USDNOK",
    "sek_df_processed": "USDSEK",
}

def latest_window_multivariate_ols(df: pd.DataFrame, y_col: str, window: int = 252):
    """Fit multivariate OLS on the most recent rolling window and return (summary, r2)."""
    if y_col not in df.columns:
        raise KeyError(f"{y_col} not found in dataframe columns.")

    window_df = df.tail(window).dropna()
    if len(window_df) < 2:
        raise ValueError("Not enough non-NaN rows in the latest rolling window.")

    y = window_df[y_col]
    X = window_df.drop(columns=[y_col])
    X_const = sm.add_constant(X, has_constant="add")

    model = sm.OLS(y, X_const).fit()
    r2 = model.rsquared

    betas = model.params.drop("const")
    pvals = model.pvalues.drop("const")
    signif_pct = (1 - pvals) * 100

    ranked_idx = betas.abs().sort_values(ascending=False).index
    betas = betas.reindex(ranked_idx)
    signif_pct = signif_pct.reindex(ranked_idx)

    summary = pd.DataFrame({
        "Driver Name": betas.index,
        "Beta Coefficient": betas.values,
        "Rank": np.arange(1, len(betas) + 1),
        "Significance %": signif_pct.values,
    })
    summary["Beta Coefficient"] = summary["Beta Coefficient"].round(4)
    summary["Significance %"] = summary["Significance %"].round(2)
    return summary, r2

def run_processed_df_regression(df_name: str, y_col: str | None = None, window: int = 252):
    """Convenience wrapper for processed_df_map by name."""
    if df_name not in processed_df_map:
        raise KeyError(f"{df_name} not found in processed_df_map.")

    if y_col is None:
        y_col = DEFAULT_Y_COL_MAP.get(df_name)
        if y_col is None:
            raise KeyError(f"No default y_col found for {df_name}.")

    summary, r2 = latest_window_multivariate_ols(processed_df_map[df_name], y_col, window=window)
    display(summary)
    print(f"Current 1Y Rolling R2: {r2:.4f}")
    return summary, r2




In [ ]:
sorted(processed_df_map.keys())


In [10]:
# Example usage (default y_col is picked automatically)
run_processed_df_regression("nok_df_processed")


,Driver Name,Beta Coefficient,Rank,Significance %
0,5y yield,3.5072,1,85.8100
1,10y yield,-2.0150,2,74.5500
2,BBDXY,0.8564,3,100.0000
3,EMFX,-0.5941,4,99.9200
4,Real 2y yield,-0.3468,5,72.8300
5,MSCI World - S&P500,0.1353,6,30.1800
6,BCOM Index,-0.1199,7,96.6500
7,Local - S&P500,0.0999,8,41.6600
8,2y yield,0.0956,9,5.9000
9,Local Index,-0.0786,10,33.1400


Current 1Y Rolling R2: 0.7317


(               Driver Name  Beta Coefficient  Rank  Significance %
 0                 5y yield            3.5072     1         85.8100
 1                10y yield           -2.0150     2         74.5500
 2                    BBDXY            0.8564     3        100.0000
 3                     EMFX           -0.5941     4         99.9200
 4            Real 2y yield           -0.3468     5         72.8300
 5      MSCI World - S&P500            0.1353     6         30.1800
 6               BCOM Index           -0.1199     7         96.6500
 7           Local - S&P500            0.0999     8         41.6600
 8                 2y yield            0.0956     9          5.9000
 9              Local Index           -0.0786    10         33.1400
 10                     Oil           -0.0677    11         99.9800
 11        Local - Wilshire           -0.0482    12         19.3200
 12         TTF Natural Gas           -0.0438    13         79.1500
 13          UK Natural Gas            0.0372   

In [20]:
def run_processed_df_regression_sig_only(
    df_name: str,
    y_col: str | None = None,
    window: int = 252,
    min_significance: float = 95.0,
):
    """Same as run_processed_df_regression but filters to >= min_significance."""
    summary, r2 = run_processed_df_regression(df_name, y_col=y_col, window=window)
    filtered = summary[summary["Significance %"] >= min_significance].reset_index(drop=True)
    display(filtered)
    print(f"Current 1Y Rolling R2: {r2:.4f}")
    return filtered, r2

run_processed_df_regression_sig_only("aud_df_processed")



,Driver Name,Beta Coefficient,Rank,Significance %
0,2y yield,3.6053,1,98.3400
1,5y yield,-3.2437,2,85.9200
2,BBDXY,-1.0952,3,100.0000
3,MSCI World - Wilshire,0.4209,4,93.5200
4,Local Index,0.3719,5,97.9600
5,Local - S&P500,-0.3420,6,96.2600
6,MSCI World - Dow Jones,0.2041,7,93.1900
7,Real 2y yield,-0.1993,8,51.2800
8,MSCI World ex-US,-0.1674,9,71.4500
9,10y yield,0.1343,10,8.5600


Current 1Y Rolling R2: 0.7728


,Driver Name,Beta Coefficient,Rank,Significance %
0,2y yield,3.6053,1,98.3400
1,BBDXY,-1.0952,3,100.0000
2,Local Index,0.3719,5,97.9600
3,Local - S&P500,-0.3420,6,96.2600
4,Coking coal,-0.0319,16,96.8300
5,G10 FX Volatility,-0.0142,17,96.2500


Current 1Y Rolling R2: 0.7728


(         Driver Name  Beta Coefficient  Rank  Significance %
 0           2y yield            3.6053     1         98.3400
 1              BBDXY           -1.0952     3        100.0000
 2        Local Index            0.3719     5         97.9600
 3     Local - S&P500           -0.3420     6         96.2600
 4        Coking coal           -0.0319    16         96.8300
 5  G10 FX Volatility           -0.0142    17         96.2500,
 np.float64(0.7728044417673012))